# 1P parameter sweep: is feedback really invisible, or just swamped?

In the LH fixed-N result (notebook 02), every simulation varies all 6
parameters at once, so each parameter's quartile-contrast response is
marginalized over the other 5 -- adding noise on top of any real-but-small
feedback signal. The **1P set** varies one parameter at a time with the
others held at fiducial, removing that marginalization noise entirely. This
notebook checks whether A_SN1/A_AGN1/A_SN2/A_AGN2 -- null in LH -- show a
real trend once that noise is gone.

**Nothing about the 1P set's exact structure (grid size, step spacing,
whether multiple seeds exist per step, or whether directory index `p1`
really is `Omega_m`) is assumed here.** Section 1 discovers it from what's
actually on disk and the 1P parameter table; section 2 verifies the
`p<N>` → parameter-name mapping empirically before trusting it.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from src import config
from src.data_io import find_snapshots
from src.params import load_1p_params
from src.onep import (
    parse_1p_label, infer_1p_parameter_names, mean_cdf_by_step, monotonic_trend,
)
from src.pipeline import run_suite
from src.sensitivity import infer_layout
from src.plotting import plot_1p_sweep

SIM_PATH_1P = config.SIM_PATH_1P
PARAMS_FILE_1P = config.PARAMS_FILE_1P
OUTPUT_DIR = config.OUTPUT_DIR

## 1. What's actually on disk?

Lists every `1P_p<N>_<M>` snapshot found and the parameter table's shape,
before assuming any grid size or seed count.

In [2]:
files_1p = find_snapshots(SIM_PATH_1P, config.SNAP, prefix="1P_")
print(f"Found {len(files_1p)} 1P snapshots at snap={config.SNAP}")

labels = sorted({f.split("1P_")[1].split("/")[0] for f in files_1p})
parsed = [parse_1p_label(l) for l in labels]
by_param = pd.Series([p for p, _ in parsed]).value_counts().sort_index()

print("\nSteps found per parameter index:")
print(by_param)

steps_per_param = pd.Series(parsed).groupby([p for p, _ in parsed]).apply(
    lambda rows: sorted({s for _, s in rows})
)
print("\nDistinct step indices per parameter:")
print(steps_per_param)

theta_1p_all = load_1p_params(PARAMS_FILE_1P)
print(f"\nParameter table: {theta_1p_all.shape[0]} rows, columns = {list(theta_1p_all.columns)}")

Found 113 1P snapshots at snap=50

Steps found per parameter index:
0     1
1     4
2     4
3     4
4     4
5     4
6     4
7     4
8     4
9     4
10    4
11    4
12    4
13    4
14    4
15    4
16    4
17    4
18    4
19    4
20    4
21    4
22    4
23    4
24    4
25    4
26    4
27    4
28    4
Name: count, dtype: int64

Distinct step indices per parameter:
0                [0]
1     [-2, -1, 1, 2]
2     [-2, -1, 1, 2]
3     [-2, -1, 1, 2]
4     [-2, -1, 1, 2]
5     [-2, -1, 1, 2]
6     [-2, -1, 1, 2]
7     [-2, -1, 1, 2]
8     [-2, -1, 1, 2]
9     [-2, -1, 1, 2]
10    [-2, -1, 1, 2]
11    [-2, -1, 1, 2]
12    [-2, -1, 1, 2]
13    [-2, -1, 1, 2]
14    [-2, -1, 1, 2]
15      [1, 2, 3, 4]
16    [-2, -1, 1, 2]
17    [-2, -1, 1, 2]
18    [-2, -1, 1, 2]
19    [-2, -1, 1, 2]
20    [-2, -1, 1, 2]
21    [-2, -1, 1, 2]
22    [-2, -1, 1, 2]
23    [-2, -1, 1, 2]
24    [-2, -1, 1, 2]
25    [-2, -1, 1, 2]
26    [-2, -1, 1, 2]
27    [-2, -1, 1, 2]
28    [-2, -1, 1, 2]
dtype: object

Parameter ta

## 2. Verify the p<N> -> parameter-name mapping

Checks which single column of the parameter table actually varies within
each `p<N>` group, rather than trusting the CAMELS p1..p6 ordering
convention blindly. Any group in `ambiguous` needs a manual look before
proceeding.

In [3]:
mapping, ambiguous = infer_1p_parameter_names(theta_1p_all, config.ALL_PARAMS)
print("Inferred p<N> -> parameter mapping:")
for pidx, name in sorted(mapping.items()):
    print(f"  p{pidx} -> {name}")

if ambiguous:
    print("\nAMBIGUOUS groups (inspect by hand):", ambiguous)
else:
    print("\nAll groups resolved cleanly.")

assert not ambiguous, "Resolve ambiguous groups before trusting the mapping below."

KeyError: "None of [Index(['Omega_m', 'sigma_8', 'A_SN1', 'A_AGN1', 'A_SN2', 'A_AGN2'], dtype='object')] are in the [columns]"

## 3. Generate fixed-N AGN summaries over the 1P set

Same `N_TARGET`/`mass_cut`/`kvals`/`rgrid` as the LH fixed-N run (notebook
02), read from its saved `.npz`, so results are directly comparable.

In [ ]:
import glob

agn_candidates = sorted(glob.glob(f"{OUTPUT_DIR}/agn_knn_snap{config.SNAP}_M{config.MASS_CUT:.0e}_n*.npz"))
assert agn_candidates, f"No LH AGN fixed-N run found in {OUTPUT_DIR} -- run notebook 02 first."
agn_data = np.load(agn_candidates[-1], allow_pickle=True)
N_TARGET = int(agn_data["nbh"][0])
print(f"N_TARGET = {N_TARGET} (from the LH AGN run)")

In [ ]:
GENERATE = True

onep_path = f"{OUTPUT_DIR}/1p_knn_snap{config.SNAP}_M{config.MASS_CUT:.0e}_n{N_TARGET}.npz"

if GENERATE:
    result = run_suite(
        n_target=N_TARGET, tracer="luminosity",
        sim_path=SIM_PATH_1P, output_dir=OUTPUT_DIR,
        dir_prefix="1P_", numeric_id=False,
    )
else:
    data = np.load(onep_path, allow_pickle=True)
    result = {k: data[k] for k in ("sim_ids", "summaries", "nbh", "rgrid", "kvals")}

sim_ids = result["sim_ids"]
summaries = result["summaries"]
rgrid = result["rgrid"]
kvals = result["kvals"]
n_k, n_r = infer_layout(kvals, rgrid)

print(f"{len(sim_ids)} 1P simulations retained at N={N_TARGET}, summary shape {summaries.shape}")
print(f"dropped: {len(files_1p) - len(sim_ids)} / {len(files_1p)} (too few eligible AGN at this N)")

## 4. Sweep plots and trend test, per parameter

In [ ]:
trend_rows = []

for pidx, name in sorted(mapping.items()):
    steps, mean_cdfs = mean_cdf_by_step(summaries, sim_ids, param_index=pidx, n_k=n_k, n_r=n_r)

    if len(steps) == 0:
        print(f"{name}: no retained simulations at N={N_TARGET}, skipping")
        continue

    step_labels = [f"p{pidx}_{s}" for s in steps]
    step_values = theta_1p_all.loc[step_labels, name].values

    order = np.argsort(step_values)
    step_values, mean_cdfs = step_values[order], mean_cdfs[order]

    fig, axes = plot_1p_sweep(rgrid, kvals, step_values, mean_cdfs, name)

    if len(step_values) >= 3:
        # trend test at the r-bin closest to where the LH Omega_m signal
        # peaked (~4 Mpc/h), k=1 -- a fixed, physically-motivated probe bin
        # rather than scanning all bins and picking the best one
        probe_r = np.argmin(np.abs(rgrid - 4.0))
        rho, p = monotonic_trend(step_values, mean_cdfs[:, 0, probe_r])
        trend_rows.append({"parameter": name, "n_steps": len(step_values),
                           "spearman_rho": rho, "p_value": p})
    else:
        print(f"{name}: only {len(step_values)} step(s) retained, too few for a trend test")

trend_df = pd.DataFrame(trend_rows)
trend_df

## Reading this

`trend_df` is a monotonicity check at one fixed, physically-motivated bin
(r~4 Mpc/h, k=1 -- where the LH `Omega_m` signal peaked), not a full
scale-resolved significance test like `sensitivity_table` -- 1P's handful of
ordered grid points isn't where a shuffle-label permutation null applies the
way it does at 1000 LH simulations. Read it as a first pass:

- **A feedback parameter with a strong, significant `rho` here** that was
  null in LH means the LH result was noise-swamped, not a real null --
  worth building a proper 1P-appropriate significance test (e.g. against
  the CV set's seed-to-seed scatter as a noise floor) before drawing
  conclusions.
- **Still no trend here either** is a much stronger case for "feedback
  really doesn't imprint on this statistic at this N/snapshot" than the LH
  result alone, since the marginalization-noise explanation is now ruled
  out.

`Omega_m`'s row is the sanity check: it should show a strong, significant
trend here too, consistent with its dominant LH signal.